# LeetCode 28: Find the Index of the First Occurrence in a String
## Difficulty: Easy

**Problem:** Given two strings `needle` and `haystack`, return the index of the first occurrence of `needle` in `haystack`, or `-1` if `needle` is not part of `haystack`.

**Constraints:**
- `1 <= haystack.length, needle.length <= 10^4`
- `haystack` and `needle` consist of only lowercase English characters.

## 1. Understanding the Problem

We need to build what Python's `.find()` does under the hood. Think of it as sliding `needle` across `haystack` one position at a time, checking for a match at each position.

```
haystack = "sadbutsad"
needle   = "sad"

Position 0: [s a d] b u t s a d  →  "sad" == "sad"?  ✓  return 0
Position 1:  s [a d b] u t s a d  →  "adb" == "sad"?  ✗
Position 2:  s a [d b u] t s a d  →  "dbu" == "sad"?  ✗
...
Position 6:  s a d b u t [s a d]  →  "sad" == "sad"?  ✓  (but we already found it at 0)
```

## 2. My Initial Approach

Before arriving at the correct solution, I went through several attempts. Here's the process and why each one fell short.

### Attempt 1: Using `in` and character matching
My first instinct was to check if `needle` exists in `haystack` using Python's `in` operator, then try to find the position by matching individual characters.

In [ ]:
# MY FIRST ATTEMPT
class SolutionAttempt1:
    def strStr(self, haystack: str, needle: str) -> int:
        if needle not in haystack:
            print(-1)
        else:
            for i in needle[0]:
                for j in haystack:
                    if i == j:
                        return i

sol = SolutionAttempt1()
result = sol.strStr("sadbutsad", "sad")
print(f"Result: {result}")

**Why this doesn't work:**

1. **`needle not in haystack` does the hard work for us** — The `in` operator already searches for the substring internally. The whole point of this problem is to implement that search logic ourselves.

2. **`print(-1)` instead of `return -1`** — Printing a value is not the same as returning it. The function returns `None` in this branch, which isn't what LeetCode expects.

3. **`for i in needle[0]`** — `needle[0]` is a single character (e.g., `"s"`). Iterating over a single character just gives you that character once. This doesn't scan through `haystack` at all.

4. **Matching individual characters isn't enough** — Finding where `"s"` appears doesn't mean `"sad"` starts there. The character `"s"` might appear in `"smile"` but `"sad"` isn't in `"smile"`. We need to match the *entire substring*, not just the first character.

5. **`return i` returns a character, not an index** — Since `i` is a character from the loop, we'd return `"s"` instead of a number like `0`.

### Attempt 2: Slicing with character loop

In [ ]:
# MY SECOND ATTEMPT
class SolutionAttempt2:
    def strStr(self, haystack: str, needle: str) -> int:
        for i in str(haystack):  # looping over characters, not indices
            if haystack[i:i+len(needle)] == needle:
                return 0  # always returning 0

# This would crash because you can't slice a string with a character:
# haystack["s":"s"+3] makes no sense
# Uncomment to see the error:
# SolutionAttempt2().strStr("sadbutsad", "sad")
print("TypeError: slice indices must be integers, not str")

**Why this doesn't work:**

1. **`for i in str(haystack)`** loops over *characters* (`"s"`, `"a"`, `"d"`, ...), not *numbers* (0, 1, 2, ...). Slicing requires integer indices — `haystack["s":"s"+3]` is a `TypeError`.

2. **`return 0`** — Even if the loop worked, this always returns 0 regardless of where the match actually is. If the match is at position 5, we'd still incorrectly say 0.

### Attempt 3: Right idea, wrong `return -1` placement

In [ ]:
# MY THIRD ATTEMPT
class SolutionAttempt3:
    def strStr(self, haystack: str, needle: str) -> int:
        for i in range(len(haystack)):
            if haystack[i:i+len(needle)] == needle:
                return 0  # still returning 0 instead of i
            else:
                return -1  # inside the loop — exits too early!

sol = SolutionAttempt3()

# Works by accident when match is at position 0
print(f"strStr('sadbutsad', 'sad') = {sol.strStr('sadbutsad', 'sad')}")  # Returns 0 (lucky)

# Fails when match is NOT at position 0
print(f"strStr('butsad', 'sad') = {sol.strStr('butsad', 'sad')}")        # Returns -1 (WRONG, should be 3)
print(f"strStr('hello', 'll') = {sol.strStr('hello', 'll')}")            # Returns -1 (WRONG, should be 2)

**Why this doesn't work for all cases:**

This is the sneakiest bug because it *works sometimes*. If the match happens to be at position 0, it returns the right answer (well, 0, which is accidentally correct). But it fails for every other case.

**The `else: return -1` is inside the loop.** This means: "if position 0 isn't a match, immediately give up and return -1." The function never checks positions 1, 2, 3, etc.

Think of it like searching for your keys — you open one drawer, don't find them, and declare "they don't exist" without checking any other drawer.

| Test Case | Expected | Got | Why |
|-----------|----------|-----|-----|
| `("sadbutsad", "sad")` | 0 | 0 | Lucky — match at position 0 |
| `("butsad", "sad")` | 3 | -1 | Position 0 is "but" ≠ "sad", so else fires and returns -1 |
| `("hello", "ll")` | 2 | -1 | Position 0 is "he" ≠ "ll", same problem |
| `("leetcode", "leeto")` | -1 | -1 | Accidentally correct — no match anywhere |

**The fix:** Move `return -1` *outside* the loop so it only triggers after checking every position.

## 3. The Key Concept — String Slicing

The `if` statement is doing all the heavy lifting. Let's break down `haystack[i:i+len(needle)]`:

- `i` is our current starting position in `haystack`
- `len(needle)` is how many characters we need to grab
- `haystack[i:i+len(needle)]` slices out a chunk the same length as `needle`

We then compare that chunk directly to `needle`. If they're equal, we found our match.

In [1]:
# Let's visualize how slicing works at each position
haystack = "sadbutsad"
needle = "sad"

print(f"haystack = '{haystack}'  (length {len(haystack)})")
print(f"needle   = '{needle}'  (length {len(needle)})")
print()

for i in range(len(haystack)):
    chunk = haystack[i:i+len(needle)]
    match = "✓ MATCH" if chunk == needle else "✗"
    print(f"  i={i}: haystack[{i}:{i+len(needle)}] = '{chunk}'  →  '{chunk}' == '{needle}'?  {match}")

haystack = 'sadbutsad'  (length 9)
needle   = 'sad'  (length 3)

  i=0: haystack[0:3] = 'sad'  →  'sad' == 'sad'?  ✓ MATCH
  i=1: haystack[1:4] = 'adb'  →  'adb' == 'sad'?  ✗
  i=2: haystack[2:5] = 'dbu'  →  'dbu' == 'sad'?  ✗
  i=3: haystack[3:6] = 'but'  →  'but' == 'sad'?  ✗
  i=4: haystack[4:7] = 'uts'  →  'uts' == 'sad'?  ✗
  i=5: haystack[5:8] = 'tsa'  →  'tsa' == 'sad'?  ✗
  i=6: haystack[6:9] = 'sad'  →  'sad' == 'sad'?  ✓ MATCH
  i=7: haystack[7:10] = 'ad'  →  'ad' == 'sad'?  ✗
  i=8: haystack[8:11] = 'd'  →  'd' == 'sad'?  ✗


## 4. The Correct Solution

In [2]:
class Solution:
    def strStr(self, haystack: str, needle: str) -> int:
        for i in range(len(haystack)):        # walk through each starting position
            if haystack[i:i+len(needle)] == needle:  # grab a chunk and compare
                return i                       # found it — return the position
        return -1                              # loop finished with no match

sol = Solution()

# Test cases
print("Test 1:", sol.strStr("sadbutsad", "sad"))   # Expected: 0
print("Test 2:", sol.strStr("leetcode", "leeto"))   # Expected: -1
print("Test 3:", sol.strStr("butsad", "sad"))       # Expected: 3
print("Test 4:", sol.strStr("hello", "ll"))         # Expected: 2
print("Test 5:", sol.strStr("a", "a"))              # Expected: 0

Test 1: 0
Test 2: -1
Test 3: 3
Test 4: 2
Test 5: 0


## 5. Small Optimization

We don't need to check positions where `needle` can't possibly fit. If `haystack` has 9 characters and `needle` has 3, the last useful starting position is index 6 (since 6 + 3 = 9).

So instead of `range(len(haystack))`, we can use `range(len(haystack) - len(needle) + 1)`.

In [ ]:
class SolutionOptimized:
    def strStr(self, haystack: str, needle: str) -> int:
        # Only check positions where needle can still fit
        for i in range(len(haystack) - len(needle) + 1):
            if haystack[i:i+len(needle)] == needle:
                return i
        return -1

sol = SolutionOptimized()

# Same test cases — same results, fewer iterations
print("Test 1:", sol.strStr("sadbutsad", "sad"))
print("Test 2:", sol.strStr("leetcode", "leeto"))
print("Test 3:", sol.strStr("butsad", "sad"))

# Show how many iterations we save
haystack = "sadbutsad"
needle = "sad"
print(f"\nOriginal loop: {len(haystack)} iterations")
print(f"Optimized loop: {len(haystack) - len(needle) + 1} iterations")
print(f"Saved: {len(needle) - 1} unnecessary checks")

## 6. Step-by-Step Trace

In [3]:
def strStr_verbose(haystack, needle):
    """Same solution but prints every step."""
    print(f"Searching for '{needle}' in '{haystack}'")
    print(f"Need to check positions 0 through {len(haystack) - len(needle)}")
    print("-" * 50)

    for i in range(len(haystack) - len(needle) + 1):
        chunk = haystack[i:i+len(needle)]
        if chunk == needle:
            print(f"  Position {i}: '{chunk}' == '{needle}'  → MATCH FOUND!")
            print(f"\n  Answer: {i}")
            return i
        else:
            print(f"  Position {i}: '{chunk}' != '{needle}'  → keep searching...")

    print(f"\n  No match found. Answer: -1")
    return -1

print("=" * 50)
strStr_verbose("sadbutsad", "sad")
print("\n" + "=" * 50)
strStr_verbose("leetcode", "leeto")
print("\n" + "=" * 50)
strStr_verbose("mississippi", "issip")

Searching for 'sad' in 'sadbutsad'
Need to check positions 0 through 6
--------------------------------------------------
  Position 0: 'sad' == 'sad'  → MATCH FOUND!

  Answer: 0

Searching for 'leeto' in 'leetcode'
Need to check positions 0 through 3
--------------------------------------------------
  Position 0: 'leetc' != 'leeto'  → keep searching...
  Position 1: 'eetco' != 'leeto'  → keep searching...
  Position 2: 'etcod' != 'leeto'  → keep searching...
  Position 3: 'tcode' != 'leeto'  → keep searching...

  No match found. Answer: -1

Searching for 'issip' in 'mississippi'
Need to check positions 0 through 6
--------------------------------------------------
  Position 0: 'missi' != 'issip'  → keep searching...
  Position 1: 'issis' != 'issip'  → keep searching...
  Position 2: 'ssiss' != 'issip'  → keep searching...
  Position 3: 'sissi' != 'issip'  → keep searching...
  Position 4: 'issip' == 'issip'  → MATCH FOUND!

  Answer: 4


4

## 7. Complexity Analysis

**Time Complexity:** `O(n × m)` where `n = len(haystack)` and `m = len(needle)`
- We check up to `n - m + 1` starting positions
- At each position, comparing the slice takes up to `m` character comparisons

**Space Complexity:** `O(m)`
- The slice `haystack[i:i+len(needle)]` creates a new string of length `m`

**Note:** There are faster algorithms (KMP, Rabin-Karp) that solve this in `O(n + m)`, but the sliding window approach is clean and sufficient for this problem's constraints (`n, m ≤ 10^4`).